# Train the U-Net (tasks 2.2–2.5)

A thin driver for `cosmicweb.train`; the logic and tests live in `src/cosmicweb/train/` and `tests/test_train.py`.

- **Runtime:** use a **GPU** runtime (Runtime → Change runtime type → T4 GPU). Note the GPU hours you use in the issue.
- **Data:** needs dataset v1, built once with `notebooks/data_pipeline.ipynb`. This notebook copies it from Drive to the VM's local disk.
- **Outputs:** each run gets a folder on Drive (`Astronova/Research/runs/<run name>/`) holding checkpoints, `metrics.csv`, `run.json` and TensorBoard logs.
- **Disconnects:** just run the notebook again. Training continues from the last finished epoch and gives the same result as an uninterrupted run.

## 1. Setup

In [ ]:
# Same steps as docs/TEAM_GUIDE.md section 7 ("Training on Colab or Kaggle").
import os
import shutil
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
BRANCH = "feature/2.2-training-pipeline"  # change to "main" once the pull request is merged

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    %cd /content
    if not Path("cosmic-web-unet").exists():
        !git clone -q https://github.com/astronova-cosmic-web/cosmic-web-unet.git
    %cd /content/cosmic-web-unet
    !git fetch -q origin && git checkout -q {BRANCH} && git pull -q --ff-only
    !pip install -q -e . astropy pyyaml
    # Do NOT reinstall torch or numpy on Colab: use the preinstalled versions.
    REPO = Path("/content/cosmic-web-unet")
    !nvidia-smi -L
else:
    REPO = Path.cwd().resolve().parent  # notebooks/ -> repo root

sys.path.insert(0, str(REPO / "src"))  # import without restarting the runtime after pip install -e
from cosmicweb.data import load_config
from cosmicweb.train import load_experiment, run_name, train

!git -C {REPO} log -1 --format="code version: %h %s"

## 2. Choose the run

The experiment file sets everything; `OVERRIDES` changes single settings without editing it. Each combination of experiment, noise, mask and seed gets its own run folder, such as `baseline_high-mid_seed0`.

In [ ]:
EXPERIMENT = REPO / "configs" / "baseline.yaml"
OVERRIDES = [
    "train.seed=0",       # baseline seeds: 0, 1, 2 (task 2.5)
    "data.noise=high",    # none / low / mid / high
    "data.mask=mid",      # none / low / mid / high
    # Quick pilot (task 2.4): uncomment to time a short run
    # "train.epochs=2", "train.limit_train_batches=100",
]

DRIVE = Path("/content/drive/MyDrive/Astronova/Research")
DATASET_ROOT = str(DRIVE / "DataSet") if IN_COLAB else os.environ.get("COSMICWEB_DATASET_ROOT")
RUNS_DIR = DRIVE / "runs" if IN_COLAB else REPO / "runs"

cfg = load_experiment(EXPERIMENT, OVERRIDES)
cfg["data"]["dataset_root"] = DATASET_ROOT
cfg["output"]["runs_dir"] = str(RUNS_DIR)

# Dataset v1 lives on Drive; training reads it from the VM's local disk (much faster).
drive_data = load_config(cfg["data"]["config"], dataset_root=DATASET_ROOT).build.output_dir
local_data = Path("/content/cosmicweb_data") / drive_data.name if IN_COLAB else drive_data
if not (local_data / "dataset.json").exists():
    if not (drive_data / "dataset.json").exists():
        raise FileNotFoundError(f"No dataset in {drive_data}: build it first with notebooks/data_pipeline.ipynb")
    print("Copying dataset v1 from Drive to local disk (a few minutes)...")
    shutil.copytree(drive_data, local_data, dirs_exist_ok=True)
cfg["data"]["root"] = str(local_data)

print("run      :", RUNS_DIR / run_name(cfg))
print("dataset  :", local_data)
print("resuming :", (RUNS_DIR / run_name(cfg) / "checkpoints" / "last.pt").exists())

## 3. Train

In [ ]:
info = train(cfg, resume="auto")  # re-run after a disconnect: continues from the last finished epoch

## 4. Results

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd

run_dir = RUNS_DIR / run_name(cfg)
m = pd.read_csv(run_dir / "metrics.csv")
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(m.epoch, m.train_loss, label="train")
ax[0].plot(m.epoch, m.val_loss, label="validation")
ax[0].set(xlabel="epoch", ylabel="MSE (scaled units)", yscale="log", title=run_dir.name)
ax[0].legend()
ax[1].plot(m.epoch, m.val_rmse_kappa)
ax[1].set(xlabel="epoch", ylabel="validation RMSE (κ)", title="validation error in physical units")
fig.tight_layout()
plt.show()

info = json.loads((run_dir / "run.json").read_text())
print(f"best epoch      : {info['best_epoch']} (val loss {info['best_val_loss']:.5f})")
print(f"epochs done     : {info['epochs_completed']} ({info['stop_reason']})")
print(f"time per epoch  : {m.epoch_time_s.median():.0f} s (median)")
print(f"peak GPU memory : {info['peak_gpu_mem_gb']} GB")
print(f"checkpoint size : {info['checkpoint_mb']} MB (last.pt)")
print(f"dataset         : {info['dataset_fingerprint'][:16]}")

In [ ]:
# Live curves for all runs (also works while training in another tab)
if IN_COLAB:
    %load_ext tensorboard
    %tensorboard --logdir {RUNS_DIR}

## Notes

- **Pilot run (task 2.4):** `run.json` and `metrics.csv` record time per epoch, peak GPU memory and checkpoint size; copy them into the issue.
- **Baseline (task 2.5):** run seeds 0, 1 and 2 with the same settings. The validation inputs are identical across seeds and runs.
- **Noise × mask grid (task 3.4):** change `data.noise` and `data.mask`; each combination gets its own run folder.
- **Checkpoints:** `last.pt` (with the optimizer state, for resuming) and `best.pt` (weights only, for evaluation). Load one with `cosmicweb.train.load_model`.